# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and explore available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Dataset description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
List available record sets in the dataset, referencing their `@id`. Then, display the available fields (columns) for each one.

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.record_sets())

if not record_sets:
    print("No record sets found in this dataset.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '<no name>')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):  # If only one field
            fields = [fields]
        elif isinstance(fields, str):
            fields = [{'@id': fields}]
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)}")
            else:
                print(f"    - {field}")
        print("")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. We reference all record sets and fields strictly by their `@id` as per Croissant.

In [ ]:
# If your dataset contains record sets, load them into DataFrames by @id
dataframes = {}

if not record_sets:
    print("No record sets found to load data.")
else:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Loading records from RecordSet @id: {rs_id}")
        records_iter = dataset.records(record_set=rs_id)
        try:
            records = list(records_iter)
            if records:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"    Loaded {len(df)} record(s) with columns: {list(df.columns)}\n")
            else:
                print("    No records found for this record set.\n")
        except Exception as e:
            print(f"    Could not load record set {rs_id}. Error: {e}\n")

Below we showcase the head (first few rows) for the first successfully loaded record set (if any). Use the `@id` to select the DataFrame.

In [ ]:
if dataframes:
    # Use the first available DataFrame for inspection
    first_rs_id = next(iter(dataframes.keys()))
    print(f"First record set loaded: {first_rs_id}")
    display(dataframes[first_rs_id].head())
else:
    print("No data loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
As an example, let's select a numeric field (by its `@id`) from a loaded record set (again, referenced by its `@id`). We'll perform filtering and normalization, and group by another field if available.

*If there are no record sets or numeric fields, this code will print a message instead.*

In [ ]:
import numpy as np

if dataframes:
    # Pick first dataframe and its record_set_id
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Working on RecordSet @id: {record_set_id}")
    
    # Try to select a likely numeric field by inspecting columns & dtypes
    numeric_candidates = []
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidates.append(col)
    if not numeric_candidates and df.shape[0] > 0:
        # Try coercing columns to float, prefer those with enough non-NA entries
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notnull().sum() > 0:
                    numeric_candidates.append(col)
            except Exception:
                continue
    if not numeric_candidates:
        print("No numeric fields found for EDA in this record set.")
    else:
        # Pick first numeric field for demonstration
        numeric_field = numeric_candidates[0]
        # If necessary, convert
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std(ddof=0) if filtered_df[numeric_field].std(ddof=0) else 1
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized values (z-score) for {numeric_field}:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by another field if exists (not the numeric one)
        group_fields = [col for col in df.columns if col != numeric_field]
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
            print(f"Grouped mean of {numeric_field} by '{group_field}':")
            display(grouped_df.head())
        else:
            print(f"No suitable group field found for grouping in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Generate a simple plot to visualize the distribution of the selected numeric field, if present.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(6,4))
    df[numeric_field].hist(bins=20, color='skyblue', edgecolor='k')
    plt.title(f"Distribution of '{numeric_field}' in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we have:  
- Loaded the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset metadata with `mlcroissant`.
- Identified and listed record sets and their fields by `@id`.
- Attempted to extract and display data from record sets, referencing entities by their Croissant `@id`.
- Performed basic data exploration and a simple numeric normalization/grouping example.
- Visualized the distribution of a numeric variable (if available).  

This approach illustrates how a Croissant-defined dataset can be programmatically explored and analyzed using transparent, identifier-respecting data science workflows.